# Ejercicio 03 — Elasticsearch: Conexión y primera búsqueda

> **Material opcional / preview Día 3.**  No es parte del onboarding obligatorio. Si querés adelantarte al Día 3, corré este notebook. Si no, lo vas a ver en su momento.

## 1. Credenciales

In [5]:
import os
from pathlib import Path
from dotenv import load_dotenv, find_dotenv
 
dotenv_path = find_dotenv(usecwd=True)
load_dotenv(dotenv_path)
PROJECT_ROOT = Path(dotenv_path).parent if dotenv_path else Path.cwd()
 
def resolve_path(value: str) -> str:
    if not value:
        return value
    p = Path(value)
    if p.is_absolute():
        return str(p)
    return str(PROJECT_ROOT / p)

ES_URL      = os.getenv('ES_URL', '')
ES_USER     = os.getenv('ES_USER', 'elastic')
ES_PASSWORD = os.getenv('ES_PASSWORD', '')
ES_INDEX    = os.getenv('ES_INDEX', 'workshop_docs')
ES_CA_CERT  = resolve_path(os.getenv('ES_CA_CERT', 'certs/ca.crt'))
 
assert ES_URL and ES_PASSWORD, 'Faltan credenciales de Elastic en .env'
assert os.path.isfile(ES_CA_CERT), f'No encuentro el cert en {ES_CA_CERT}'
print('Credenciales OK')
print(f'  Cert: {ES_CA_CERT}')

Credenciales OK
  Cert: /app/certs/ca.crt


## 2. Conectar al cluster

Usamos `ca_certs` apuntando al certificado del cluster (montado en `/app/certs` dentro del contenedor).

In [6]:
from elasticsearch import Elasticsearch

es = Elasticsearch(
    ES_URL,
    basic_auth=(ES_USER, ES_PASSWORD),
    ca_certs=ES_CA_CERT
)

if es.ping():
    print('Conectado al cluster')
else:
    print('No se pudo conectar. Verificá ES_URL, credenciales y cert.')

Conectado al cluster


## 3. Info del cluster

In [7]:
info = es.info()
print('Cluster:', info['cluster_name'])
print('Versión:', info['version']['number'])

Cluster: 99c40cdb-d194-400e-8874-4808820bfd24
Versión: 8.19.0


## 4. Estructura del índice del workshop

In [8]:
mappings = es.indices.get_mapping(index=ES_INDEX)
props = mappings[ES_INDEX]['mappings'].get('properties', {})

print(f'Campos del índice "{ES_INDEX}":')
print('-' * 40)
for campo, config in props.items():
    tipo = config.get('type', 'object')
    dims = config.get('dims', '')
    dims_str = f' ({dims} dims)' if dims else ''
    print(f'  {campo:<20} {tipo}{dims_str}')

count = es.count(index=ES_INDEX)['count']
print(f'\nDocumentos indexados: {count}')

Campos del índice "workshop_docs":
----------------------------------------
  contenido            text
  embedding            dense_vector (384 dims)
  fuente               keyword

Documentos indexados: 10


## 5. Documento de ejemplo

In [9]:
resultado = es.search(index=ES_INDEX, body={'query': {'match_all': {}}, 'size': 1})
doc = resultado['hits']['hits'][0]
print('ID:', doc['_id'])
for k, v in doc['_source'].items():
    if k == 'embedding':
        print(f'  embedding: [{v[0]:.4f}, {v[1]:.4f}, ...] ({len(v)} dims)')
    else:
        val = str(v)[:100] + '...' if len(str(v)) > 100 else v
        print(f'  {k}: {val}')

ID: 1
  contenido: El error de autenticación ocurre cuando el token JWT está expirado o es inválido.
  fuente: manual_seguridad.pdf
  embedding: [0.0100, 0.0100, ...] (384 dims)


## 6. Búsqueda por keyword (BM25)

In [10]:
query_texto = 'error de autenticación'

resultado = es.search(
    index=ES_INDEX,
    body={
        'query': {'match': {'contenido': {'query': query_texto, 'operator': 'or'}}},
        'size': 3,
        '_source': ['contenido', 'fuente']
    }
)

hits = resultado['hits']['hits']
print(f'Query: "{query_texto}"')
print(f'Resultados: {len(hits)}')
print('-' * 50)
for i, hit in enumerate(hits, 1):
    print(f'\n[{i}] Score: {hit["_score"]:.4f}')
    print(f'    Fuente: {hit["_source"].get("fuente", "N/A")}')
    print(f'    Contenido: {hit["_source"].get("contenido", "")[:200]}...')

Query: "error de autenticación"
Resultados: 3
--------------------------------------------------

[1] Score: 3.7510
    Fuente: manual_seguridad.pdf
    Contenido: El error de autenticación ocurre cuando el token JWT está expirado o es inválido....

[2] Score: 1.6868
    Fuente: guia_ops.pdf
    Contenido: Los logs de autenticación se encuentran en /var/log/auth.log en el servidor principal....

[3] Score: 0.2027
    Fuente: guia_deploy.pdf
    Contenido: El deploy en producción requiere aprobación de dos revisores y pasar el pipeline de CI....
